#### Quickstart: Compare runs, choose a model, and deploy it to a REST API
In this quickstart you will:
- Run a hypermapameter sweep on a training script.
- Compare the results of the runs in the MLflow UI.
- Choose the best run and register it as a model.
- Deploy the model to a REST API.
- Build a container image suitable for deployment to a cloud platform.

In [43]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"  # 0 = all, 1 = hide INFO, 2 = hide WARNING too, 3 = errors only

import keras
import mlflow
import mlflow.tensorflow  # pyright: ignore
import numpy as np
import pandas as pd
import tensorflow as tf
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from mlflow.models import (
    convert_input_example_to_serving_input,
    infer_signature,
    validate_serving_input,
)
from sklearn.model_selection import train_test_split

In [20]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [21]:
## load the dataset
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep = ";"
)

In [22]:
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [23]:
## Split the data into training, validation and test sets
train, test = train_test_split(data, test_size=0.25, random_state=42)

In [24]:
train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
2835,6.3,0.25,0.22,3.30,0.048,41.0,161.0,0.99256,3.16,0.50,10.5,6
1157,7.8,0.30,0.29,16.85,0.054,23.0,135.0,0.99980,3.16,0.38,9.0,6
744,7.4,0.38,0.27,7.50,0.041,24.0,160.0,0.99535,3.17,0.43,10.0,5
1448,7.4,0.16,0.49,1.20,0.055,18.0,150.0,0.99170,3.23,0.47,11.2,6
3338,7.2,0.27,0.28,15.20,0.046,6.0,41.0,0.99665,3.17,0.39,10.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4426,6.2,0.21,0.52,6.50,0.047,28.0,123.0,0.99418,3.22,0.49,9.9,6
466,7.0,0.14,0.32,9.00,0.039,54.0,141.0,0.99560,3.22,0.43,9.4,6
3092,7.6,0.27,0.52,3.20,0.043,28.0,152.0,0.99129,3.02,0.53,11.4,6
3772,6.3,0.24,0.29,13.70,0.035,53.0,134.0,0.99567,3.17,0.38,10.6,6


In [25]:
train[['quality']].values

array([[6],
       [6],
       [5],
       ...,
       [6],
       [6],
       [8]], shape=(3673, 1))

In [27]:
train[['quality']].values.ravel()

array([6, 6, 5, ..., 6, 6, 8], shape=(3673,))

In [28]:
train_x = train.drop(['quality'], axis=1).values
train_y = train[['quality']].values.ravel()

## test dataset
test_x = test.drop(['quality'], axis=1).values
test_y = test[['quality']].values.ravel()

## splitting this train data into train and validation
train_x, valid_x, train_y, valid_y = train_test_split(train_x, train_y, test_size=0.20,
                                                      random_state=42)

signature = infer_signature(train_x, train_y)

In [29]:
np.mean(train_x, axis=0)

array([6.86621852e+00, 2.80377808e-01, 3.32597005e-01, 6.42164738e+00,
       4.55513955e-02, 3.53556841e+01, 1.38792376e+02, 9.94074221e-01,
       3.18919333e+00, 4.88396869e-01, 1.05005673e+01])

In [30]:
train_x.shape[1]

11

In [34]:
## ANN Model

def train_model(params, epochs, train_x, train_y, valid_x, valid_y, test_x, test_y):
    
    ## Define the model architecture
    mean = np.mean(train_x, axis=0)
    var = np.var(train_x, axis=0)

    model = keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean, variance=var),
            keras.layers.Dense(64, activation='relu'),
            keras.layers.Dense(1),
        ]
    )

    ## compile the model
    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=params["lr"],
            momentum=params["momentum"],
            ),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()],
        )
    
    ## Train the ANN model with lr and momentum params and track with MLFLOW tracking
    with mlflow.start_run(nested=True):
        model.fit(train_x, train_y, validation_data=(valid_x, valid_y),
                  epochs=epochs,
                  batch_size=64,
                  )

        ## Evaluate the model
        eval_result = model.evaluate(valid_x, valid_y, batch_size=64)
        eval_rmse = eval_result[1]

        ## Log the parameters and results
        mlflow.log_params(params=params)
        mlflow.log_metric("eval_rmse", eval_rmse)

        ## Log the model        
        mlflow.tensorflow.log_model(model, name="model", signature=signature)  # pyright: ignore[reportPrivateImportUsage]


        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

In [35]:
def objective(params):
    # MLflow will track the parameters and results from each run
    result = train_model(
        params=params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y,
    )
    return result

In [36]:
space = {
    "lr": hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum": hp.uniform("momentum", 0.0, 1.0),
}

In [38]:
mlflow.set_experiment("/wine-quality")
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials,
    )
    assert best is not None  # fmin always returns a dict when return_argmin=True

    # Fetch the details of the best run
    best_run = min(trials.results, key=lambda x: x["loss"])

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], name="model", signature=signature) # pyright: ignore[reportPrivateImportUsage]

# Print out the best parameters and corresponding loss
print(f"Best parameters: {best}")
print(f"Best eval rmse: {best_run['loss']}")

2026/11/12 23:50:14 INFO mlflow.tracking.fluent: Experiment with name '/wine-quality' does not exist. Creating a new experiment.


  0%|          | 0/4 [00:00<?, ?trial/s, best loss=?]

I0000 00:00:1794523814.734476   36740 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2859 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1050 Ti, pci bus id: 0000:07:00.0, compute capability: 6.1


Epoch 1/3                                            

  0%|          | 0/4 [00:04<?, ?trial/s, best loss=?]

I0000 00:00:1794523819.184284  127539 service.cc:153] XLA service 0x74c0b0035250 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1794523819.184312  127539 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce GTX 1050 Ti, Compute Capability 6.1 (Driver: 12.6.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.28.1)
I0000 00:00:1794523819.218455  127539 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1794523819.277950  127539 cuda_dnn.cc:461] Loaded cuDNN version 92801
I0000 00:00:1794523819.294896  127539 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_659__.8


 1/46 ━━━━━━━━━━━━━━━━━━━━ 43s 975ms/step - loss: 32.3734 - root_mean_squared_error: 5.6898
25/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.0047 - root_mean_squared_error: 2.0012    
  0%|          | 0/4 [00:05<?, ?trial/s, best loss=?]

... (10 outputs left out)

I0000 00:00:1794523844.138484  127539 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_6035__.8


46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 25.0030 - root_mean_squared_error: 5.0003
 75%|███████▌  | 3/4 [00:30<00:08,  8.81s/trial, best loss: 0.7347486615180969]

I0000 00:00:1794523844.383255  127538 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_6035__.8


46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 25.0030 - root_mean_squared_error: 5.0003 - val_loss: 16.8196 - val_root_mean_squared_error: 4.1012

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 18.3961 - root_mean_squared_error: 4.2891
30/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 13.6305 - root_mean_squared_error: 3.6919 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 12.0559 - root_mean_squared_error: 3.4722 - val_loss: 8.0238 - val_root_mean_squared_error: 2.8326

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 6.8766 - root_mean_squared_error: 2.6223
31/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 6.3711 - root_mean_squared_error: 2.5241 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 6.0057 - root_mean_squared_error: 2.4506 - val_loss: 4.3768 - val_root_mean_squared_error: 2.0921

 1/12 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/ste

In [ ]:
# $ uv run mlflow ui

#### We are now going to validate the model with Claude's updated code.

In [39]:
model_info = mlflow.tensorflow.log_model(
    best_run["model"], name="model", signature=signature
)  # pyright: ignore[reportPrivateImportUsage]

In [42]:
model_info.model_uri

'models:/m-7af1afd2170540a3ba74619bed57b53c'

In [44]:
serving_payload = convert_input_example_to_serving_input(test_x[:5])
assert serving_payload is not None
print(validate_serving_input(model_info.model_uri, serving_payload))

/tmp/ipykernel_36740/393440336.py:3: FutureWarning: ``mlflow.models.utils.validate_serving_input`` is deprecated since 3.13.0. This method will be removed in a future release. Use ``mlflow.models.predict`` instead.
  print(validate_serving_input(model_info.model_uri, serving_payload))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step
[[5.862955 ]
 [6.815114 ]
 [6.6070127]
 [5.884103 ]
 [6.299919 ]]


In [45]:
## Another way more updated to do it using `mlflow.models.predict`
mlflow.models.predict(
    model_uri=model_info.model_uri,
    input_data=test_x[:5],
    env_manager="local",
)

2026/11/13 00:47:42 INFO mlflow.models.python_api: It is highly recommended to use `uv` as the environment manager for predicting with MLflow models as its performance is significantly better than other environment managers. Run `pip install uv` to install uv. See https://docs.astral.sh/uv/getting-started/installation for other installation methods.
2026/11/13 00:47:42 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
{"predictions": [[5.862955093383789], [6.8151140213012695], [6.607012748718262], [5.884102821350098], [6.299919128417969]]}

In [46]:
## If you want the predictions as a value you can work with, load the model through MLflow's generic interface instead:

loaded = mlflow.pyfunc.load_model(model_info.model_uri)
preds = loaded.predict(test_x[:5])
preds

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step


array([[5.862955 ],
       [6.815114 ],
       [6.6070127],
       [5.884103 ],
       [6.299919 ]], dtype=float32)

In [50]:
"""
- "uv" is what the UI suggests. It builds a fresh environment from the model's requirements.txt and predicts there, which is the most realistic pre-deployment check. For your model that means downloading TensorFlow and the CUDA packages again (several GB, a few minutes) the first time.
- "local" reuses your current .venv. It's instant, but it only proves the model works in your environment.
"""

## the most similar code to whatr the teacher is using to validate the model

model_uri = "models:/m-709b7427d1994f889a8eb62da8c11325"  # or model_info.model_uri

# Replace with your own input example: a valid input for pyfunc prediction
input_data = test_x[:5]

# Verify the model with the provided input data using the logged dependencies.
# https://mlflow.org/docs/latest/models.html#validate-models-before-deployment
mlflow.models.predict(
    model_uri=model_uri,
    input_data=input_data,
    # env_manager="uv",
    env_manager="local",
)

2026/11/13 01:03:22 INFO mlflow.models.python_api: It is highly recommended to use `uv` as the environment manager for predicting with MLflow models as its performance is significantly better than other environment managers. Run `pip install uv` to install uv. See https://docs.astral.sh/uv/getting-started/installation for other installation methods.
2026/11/13 01:03:22 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
{"predictions": [[5.862955093383789], [6.8151140213012695], [6.607012748718262], [5.884102821350098], [6.299919128417969]]}

In [51]:
# If you also want the UI's "with input example" variant, log the model with input_example=train_x[:5]. Then you can use input_data = mlflow.pyfunc.load_model(model_uri).input_example instead of supplying data yourself.

#### Finally we validate the model with the teacher's code from the old mlflow version

In [52]:
## Inferencing

model_info = mlflow.tensorflow.log_model(
    best_run["model"], name="model", signature=signature
)

model_uri = model_info.model_uri

# The logged model does not contain an input_example.
# Manually generate a serving payload to verify your model prior to deployment.
from mlflow.models import convert_input_example_to_serving_input

# Define INPUT_EXAMPLE via assignment with your own input example to the model
# A valid input example is a data instance suitable for pyfunc prediction
# serving_payload = convert_input_example_to_serving_input(INPUT_EXAMPLE)
serving_payload = convert_input_example_to_serving_input(test_x)

# Validate the serving payload works on the model
assert serving_payload is not None
validate_serving_input(model_uri, serving_payload)

/tmp/ipykernel_36740/3926493728.py:20: FutureWarning: ``mlflow.models.utils.validate_serving_input`` is deprecated since 3.13.0. This method will be removed in a future release. Use ``mlflow.models.predict`` instead.
  validate_serving_input(model_uri, serving_payload)


39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


array([[5.862955 ],
       [6.815114 ],
       [6.6070127],
       ...,
       [6.3010645],
       [6.8637075],
       [5.958558 ]], shape=(1225, 1), dtype=float32)

In [53]:
## Another way recommended by the teacher using PyFuncModel

# Load model as a PyFuncModel.
logged_model = model_info.model_uri   # e.g. 'models:/m-709b7427d1994f889a8eb62da8c11325'
loaded_model = mlflow.pyfunc.load_model(logged_model)

# Predict on a Pandas DataFrame.
loaded_model.predict(pd.DataFrame(test_x))

39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


array([[5.862955 ],
       [6.815114 ],
       [6.6070127],
       ...,
       [6.3010645],
       [6.8637075],
       [5.958558 ]], shape=(1225, 1), dtype=float32)

In [54]:
# To see how good the predictions are, compare them with the true quality scores:

preds = loaded_model.predict(test_x).ravel()
rmse = np.sqrt(np.mean((preds - test_y) ** 2))
print(f"Test RMSE: {rmse:.4f}")

39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Test RMSE: 0.7459


In [55]:
## Register in the model registry
mlflow.register_model(model_uri=model_info.model_uri, name="wine-quality")

Successfully registered model 'wine-quality'.
Created version '1' of model 'wine-quality'.


<ModelVersion: aliases=[], creation_timestamp=1794529427157, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1794529427157, metrics=None, model_id=None, name='wine-quality', params=None, run_id=None, run_link=None, source='models:/m-b29759b70ce543e3a9eb0ed1374d2f88', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>